In [8]:
!pip install -q -U langchain-text-splitters chromadb sentence-transformers

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",

    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()

try:
    client.delete_collection("rag_chunks")
except Exception:
    pass

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)

    collection.add(
        documents=doc_chunks,
        metadatas=[
            {
                "source": source,
                "chunk_index": i,
                "char_count": len(chunk)
            }
            for i, chunk in enumerate(doc_chunks)
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(doc_chunks))
        ]
    )

print("Total chunks stored:", collection.count())

user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print("\nBasic Retrieval Results:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk: {meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

def distance_to_similarity(distance):
    return 1 - distance

print("\nSimilarity Scores:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    similarity = distance_to_similarity(distance)

    print(
        f"Similarity: {similarity:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk: {meta['chunk_index']}"
    )

def retrieve_with_threshold(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.3
):
    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            filtered.append({
                "text": doc,
                "metadata": meta,
                "similarity": similarity
            })

    return filtered

relevant_matches = retrieve_with_threshold(
    "What is deep learning?",
    collection,
    n_results=5,
    similarity_threshold=0.3
)

print("\nRelevant Query:")

print(
    f"Chunks passing threshold: "
    f"{len(relevant_matches)}"
)

for match in relevant_matches:
    print(
        f"Similarity: {match['similarity']:.4f} | "
        f"Source: {match['metadata']['source']}"
    )
    print(match["text"])
    print("-" * 60)

unrelated_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    n_results=5,
    similarity_threshold=0.3
)

print("\nUnrelated Query:")

print(
    f"Chunks passing threshold: "
    f"{len(unrelated_matches)}"
)

if len(unrelated_matches) == 0:
    print(
        "No relevant context found. "
        "The RAG system should avoid answering from unrelated documents."
    )

results = collection.query(
    query_texts=[
        "How does search work using meaning instead of keywords?"
    ],
    n_results=2,
    where={
        "source": "embeddings_in_practice.txt"
    },
    include=["documents", "metadatas", "distances"]
)

print("\nMetadata Filtered Retrieval:")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    similarity = distance_to_similarity(distance)

    print(
        f"Similarity: {similarity:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk: {meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None
):
    query_kwargs = {
        "query_texts": [query],
        "n_results": min(n_results, collection.count()),
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if source_filter:
        query_kwargs["where"] = {
            "source": source_filter
        }

    results = collection.query(**query_kwargs)

    context_chunks = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return context_chunks

context = retrieve_context(
    "What is machine learning?",
    collection,
    n_results=3,
    similarity_threshold=0.25
)

print("\nReusable Retrieval Function:")

for item in context:
    print(
        f"Similarity: {item['similarity']} | "
        f"Source: {item['source']} | "
        f"Chunk: {item['chunk_index']}"
    )
    print(item["text"])
    print("-" * 60)

def format_context_for_llm(query, context_chunks):
    if not context_chunks:
        return (
            f"Question: {query}\n\n"
            "No relevant context was found in the knowledge base."
        )

    context_text = "\n\n".join(
        f"[Source: {item['source']}, "
        f"Chunk: {item['chunk_index']}]\n"
        f"{item['text']}"
        for item in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not present in the context, say you don't know.

Context:

{context_text}

Question: {query}

Answer:"""

    return prompt

query = "How does RAG combine search with a language model?"

context = retrieve_context(
    query,
    collection,
    n_results=3,
    similarity_threshold=0.25
)

final_prompt = format_context_for_llm(
    query,
    context
)

print("\nLLM-Ready Prompt:")
print(final_prompt)

test_queries = [
    "Why do companies use vector databases?",
    "What is the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?"
]

print("\nParaphrased Query Testing:")

for query in test_queries:
    print(f"\nQuery: {query}")

    matches = retrieve_context(
        query,
        collection,
        n_results=2,
        similarity_threshold=0.25
    )

    if not matches:
        print("No relevant context found.")
    else:
        for match in matches:
            print(
                f"Similarity: {match['similarity']} | "
                f"Source: {match['source']}"
            )
            print(match["text"][:150])
            print()

print("\nStep 3 of RAG completed successfully!")
print("Total chunks in ChromaDB:", collection.count())

Total chunks stored: 13

Basic Retrieval Results:
Distance: 0.3244 | Source: ai_overview.txt | Chunk: 6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
Distance: 0.6356 | Source: ai_overview.txt | Chunk: 7
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------
Distance: 0.6794 | Source: embeddings_in_practice.txt | Chunk: 2
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------

Similarity Scores:
Similarity: 0.6756 | Source: ai_overview.txt | Chunk: 6
Similarity: 0.3644 | Source: ai_overview.txt | Chunk: 7
Sim